In [1]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report

# 1. Load Data
dim_stores = pd.read_csv('dim_stores.csv')
dim_skus = pd.read_csv('dim_skus.csv')
dim_suppliers = pd.read_csv('dim_suppliers.csv', na_values=['N/A'], keep_default_na=True)
dim_events = pd.read_csv('dim_events.csv', parse_dates=['date'])
fact = pd.read_csv('fact_inventory_daily.csv', parse_dates=['date'])

# 2. Clean & Merge Tables
dim_stores['city_display'] = dim_stores['city_display'].str.title()
df = fact.merge(dim_stores, on='store_id', how='left')
df = df.merge(dim_skus.drop(columns=['supplier_id']), on='sku_id', how='left')
df = df.merge(dim_suppliers, on='supplier_id', how='left')
df = df.merge(dim_events, on='date', how='left')

# Contextual Imputation for missing reliability scores
category_medians = df.groupby('category')['reliability_score'].transform('median')
df['supplier_reliability_clean'] = df['reliability_score'].fillna(category_medians)

# 3. Feature Engineering
df = df.sort_values(by=['store_id', 'sku_id', 'date']).reset_index(drop=True)

# Core inventory metrics
df['reorder_gap'] = df['reorder_point'] - df['closing_stock']
df['days_of_cover_ratio'] = df['days_of_cover'] / df['lead_time_days_expected']

# Rolling 3-day recent reorder flag
df['reorder_placed_num'] = df['reorder_placed'].map({'Y': 1, 'N': 0})
df['is_recent_reorder'] = df.groupby(['store_id', 'sku_id'])['reorder_placed_num'].shift(1).rolling(3, min_periods=1).max().fillna(0)

# Temporal features & boolean mapping
df['day_of_month'] = df['date'].dt.day
festival_start = pd.to_datetime('2026-10-22')
df['days_since_festival_start'] = (df['date'] - festival_start).dt.days
df['is_perishable'] = df['is_perishable'].map({'Y': 1, 'N': 0})
df['festive_relevant'] = df['festive_relevant'].map({'Y': 1, 'N': 0})

# Target mapping
df['target'] = df['stockout_risk'].map({'Safe': 0, 'At-Risk': 1, 'Imminent': 2})

# 4. Final Prep & Temporal Split
features = [
    'reorder_gap', 'days_of_cover_ratio', 'supplier_reliability_clean',
    'is_recent_reorder', 'day_of_month', 'days_since_festival_start',
    'sales_velocity_7d', 'is_perishable', 'festive_relevant'
]

# One-hot encode product categories
df = pd.get_dummies(df, columns=['category'], drop_first=True)
features += [col for col in df.columns if col.startswith('category_')]

X = df[features]
y = df['target']

# Strict temporal split to test true generalization
train_mask = df['date'] <= '2026-10-23'
test_mask = df['date'] > '2026-10-23'

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

# 5. Train & Evaluate Model
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)

y_pred = gb.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Safe', 'At-Risk', 'Imminent']))

              precision    recall  f1-score   support

        Safe       0.95      1.00      0.97      3139
     At-Risk       0.84      0.79      0.82      1126
    Imminent       0.90      0.79      0.84       775

    accuracy                           0.92      5040
   macro avg       0.90      0.86      0.88      5040
weighted avg       0.92      0.92      0.92      5040

